In [5]:
#Checking versions of each library

import pandas as pd
import numpy as np
import sklearn

print("Pandas version:", pd.__version__)
print("NumPy version:", np.__version__)
print("Scikit-learn version:", sklearn.__version__)

Pandas version: 3.0.3
NumPy version: 2.4.6
Scikit-learn version: 1.9.0


In [6]:
from pathlib import Path

data_file = Path(
    "/Users/kevinvillon/Desktop/NOVA_IT691/Data_Version2/cicids2017_cleaned_WHOLE_SET_V2.csv"
)

print("File exists:", data_file.exists())
print("File:", data_file)

File exists: True
File: /Users/kevinvillon/Desktop/NOVA_IT691/Data_Version2/cicids2017_cleaned_WHOLE_SET_V2.csv


In [7]:
# Small test load before loading the entire dataset

sample_df = pd.read_csv(data_file, nrows=5000)

print("Sample shape:", sample_df.shape)
sample_df.head()

Sample shape: (5000, 53)


,Destination Port,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,...,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Max,Active Min,Idle Mean,Idle Max,Idle Min,Attack Type
0,22,1266342,41,2664,456,0,64.975610,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
1,22,1319353,41,2664,456,0,64.975610,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
2,22,160,1,0,0,0,0.000000,0.000000,0,0,...,243,0,32,0.0,0,0,0.0,0,0,Normal Traffic
3,22,1303488,41,2728,456,0,66.536585,110.129945,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
4,35396,77,1,0,0,0,0.000000,0.000000,0,0,...,290,0,32,0.0,0,0,0.0,0,0,Normal Traffic


In [8]:
# Load the entire V2 dataset

df = pd.read_csv(data_file)

print("Full dataset loaded.")
print("Rows:", df.shape[0])
print("Columns:", df.shape[1])

Full dataset loaded.
Rows: 2520751
Columns: 53


In [9]:
# Inspect column names

print(df.columns.tolist())

['Destination Port', 'Flow Duration', 'Total Fwd Packets', 'Total Length of Fwd Packets', 'Fwd Packet Length Max', 'Fwd Packet Length Min', 'Fwd Packet Length Mean', 'Fwd Packet Length Std', 'Bwd Packet Length Max', 'Bwd Packet Length Min', 'Bwd Packet Length Mean', 'Bwd Packet Length Std', 'Flow Bytes/s', 'Flow Packets/s', 'Flow IAT Mean', 'Flow IAT Std', 'Flow IAT Max', 'Flow IAT Min', 'Fwd IAT Total', 'Fwd IAT Mean', 'Fwd IAT Std', 'Fwd IAT Max', 'Fwd IAT Min', 'Bwd IAT Total', 'Bwd IAT Mean', 'Bwd IAT Std', 'Bwd IAT Max', 'Bwd IAT Min', 'Fwd Header Length', 'Bwd Header Length', 'Fwd Packets/s', 'Bwd Packets/s', 'Min Packet Length', 'Max Packet Length', 'Packet Length Mean', 'Packet Length Std', 'Packet Length Variance', 'FIN Flag Count', 'PSH Flag Count', 'ACK Flag Count', 'Average Packet Size', 'Subflow Fwd Bytes', 'Init_Win_bytes_forward', 'Init_Win_bytes_backward', 'act_data_pkt_fwd', 'min_seg_size_forward', 'Active Mean', 'Active Max', 'Active Min', 'Idle Mean', 'Idle Max', 'Id

In [10]:
# Look for the label / attack classification column

possible_label_columns = [
    c for c in df.columns
    if c.lower() in [
        "label",
        "class",
        "target",
        "attack",
        "attack type",
        "attack_type"
    ]
]

print("Possible label columns:", possible_label_columns)

Possible label columns: ['Attack Type']


In [11]:
df.head()

,Destination Port,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,...,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Max,Active Min,Idle Mean,Idle Max,Idle Min,Attack Type
0,22,1266342,41,2664,456,0,64.975610,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
1,22,1319353,41,2664,456,0,64.975610,109.864573,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
2,22,160,1,0,0,0,0.000000,0.000000,0,0,...,243,0,32,0.0,0,0,0.0,0,0,Normal Traffic
3,22,1303488,41,2728,456,0,66.536585,110.129945,976,0,...,243,24,32,0.0,0,0,0.0,0,0,Normal Traffic
4,35396,77,1,0,0,0,0.000000,0.000000,0,0,...,290,0,32,0.0,0,0,0.0,0,0,Normal Traffic


In [12]:
# Define target label and feature columns

label_col = "Attack Type"

feature_cols = [
    col for col in df.columns
    if col != label_col
]

print("Label column:", label_col)
print("Number of feature columns:", len(feature_cols))

Label column: Attack Type
Number of feature columns: 52


In [13]:
# Check class distribution before additional cleaning

class_counts = df[label_col].value_counts(dropna=False)

class_percentages = (
    df[label_col].value_counts(normalize=True, dropna=False) * 100
)

class_distribution = pd.DataFrame({
    "Count": class_counts,
    "Percentage": class_percentages
})

class_distribution

,Count,Percentage
Attack Type,,
Normal Traffic,2095057,83.112414
DoS,193745,7.686003
DDoS,128014,5.078407
Port Scanning,90694,3.597896
Brute Force,9150,0.362987
Web Attacks,2143,0.085014
Bots,1948,0.077279


In [14]:
# Check for missing values

missing_counts = df.isna().sum()
missing_counts = missing_counts[missing_counts > 0]

print("Total missing values:", df.isna().sum().sum())

if len(missing_counts) == 0:
    print("No missing values found.")
else:
    print("\nColumns with missing values:")
    print(missing_counts)

Total missing values: 0
No missing values found.


In [15]:
# Check numerical feature columns for positive or negative infinity

numeric_cols = df[feature_cols].select_dtypes(include=np.number).columns

inf_counts = {}

for col in numeric_cols:
    count = np.isinf(df[col].to_numpy()).sum()

    if count > 0:
        inf_counts[col] = count

print("Columns containing infinity values:")

if len(inf_counts) == 0:
    print("No infinity values found.")
else:
    for col, count in inf_counts.items():
        print(col, ":", count)

Columns containing infinity values:
No infinity values found.


In [18]:
# Identify exact duplicate records
# Exact duplicate = all 52 features AND Attack Type are identical

exact_duplicate_mask = df.duplicated(
    subset=df.columns,
    keep=False
)

extra_duplicate_mask = df.duplicated(
    subset=df.columns,
    keep="first"
)

print(
    "Rows that belong to exact duplicate groups:",
    exact_duplicate_mask.sum()
)

print(
    "Extra exact duplicate rows:",
    extra_duplicate_mask.sum()
)

Rows that belong to exact duplicate groups: 322
Extra exact duplicate rows: 161


In [19]:
df.loc[exact_duplicate_mask].head(20)

,Destination Port,Flow Duration,Total Fwd Packets,Total Length of Fwd Packets,Fwd Packet Length Max,Fwd Packet Length Min,Fwd Packet Length Mean,Fwd Packet Length Std,Bwd Packet Length Max,Bwd Packet Length Min,...,Init_Win_bytes_backward,act_data_pkt_fwd,min_seg_size_forward,Active Mean,Active Max,Active Min,Idle Mean,Idle Max,Idle Min,Attack Type
1391,80,49,2,12,6,6,6.0,0.0,0,0,...,-1,1,20,0.0,0,0,0.0,0,0,Normal Traffic
1757,21,1,2,12,6,6,6.0,0.0,0,0,...,-1,1,20,0.0,0,0,0.0,0,0,Normal Traffic
12549,443,4,2,0,0,0,0.0,0.0,0,0,...,-1,0,32,0.0,0,0,0.0,0,0,Normal Traffic
13968,443,47,2,0,0,0,0.0,0.0,0,0,...,-1,0,32,0.0,0,0,0.0,0,0,Normal Traffic
17719,443,3,2,0,0,0,0.0,0.0,0,0,...,-1,0,32,0.0,0,0,0.0,0,0,Normal Traffic
18157,21,1,2,12,6,6,6.0,0.0,0,0,...,-1,1,20,0.0,0,0,0.0,0,0,Normal Traffic
26825,443,48,2,0,0,0,0.0,0.0,0,0,...,-1,0,32,0.0,0,0,0.0,0,0,Normal Traffic
26963,21,1,2,12,6,6,6.0,0.0,0,0,...,-1,1,20,0.0,0,0,0.0,0,0,Normal Traffic
27685,443,3,2,0,0,0,0.0,0.0,0,0,...,-1,0,32,0.0,0,0,0.0,0,0,Normal Traffic
34830,21,1,2,12,6,6,6.0,0.0,0,0,...,-1,1,20,0.0,0,0,0.0,0,0,Normal Traffic


In [20]:
df.loc[exact_duplicate_mask, label_col].value_counts()

Attack Type
Normal Traffic    322
Name: count, dtype: int64

In [21]:
# Create a hash representing each 52-feature vector

feature_hash = pd.util.hash_pandas_object(
    df[feature_cols],
    index=False
)

print("Feature hashes created:", len(feature_hash))

Feature hashes created: 2520751


In [22]:
# Find feature vectors that occur multiple times

repeated_feature_mask = feature_hash.duplicated(
    keep=False
)

print(
    "Rows whose 52-feature vector appears more than once:",
    repeated_feature_mask.sum()
)

print(
    "Unique repeated feature-vector groups:",
    feature_hash[repeated_feature_mask].nunique()
)

Rows whose 52-feature vector appears more than once: 1716
Unique repeated feature-vector groups: 858


In [24]:
# Check whether identical feature vectors have conflicting Attack Type labels

audit_df = pd.DataFrame({
    "feature_hash": feature_hash,
    "Attack Type": df[label_col]
})

label_counts_per_vector = (
    audit_df
    .groupby("feature_hash")[label_col]
    .nunique()
)

conflicting_hashes = label_counts_per_vector[
    label_counts_per_vector > 1
].index

conflicting_mask = feature_hash.isin(conflicting_hashes)

print(
    "Feature-vector groups with conflicting labels:",
        len(conflicting_hashes)
)

print(
    "Total rows involved in conflicting-label groups:",
    conflicting_mask.sum()
)

Feature-vector groups with conflicting labels: 697
Total rows involved in conflicting-label groups: 1394


In [ ]:
# Verify repeated feature vectors using the actual 52 feature columns

repeated_rows = df.loc[
    repeated_feature_mask,
    feature_cols + [label_col]
].copy()

exact_feature_groups = (
    repeated_rows
    .groupby(feature_cols, dropna=False)[label_col]
    .agg(
        group_size="size",
        number_of_labels="nunique"
    )
    .reset_index()
)

print(
    "Repeated feature-vector groups:",
    len(exact_feature_groups)
)
print(
    "Same-feature / same-label groups:",
    (exact_feature_groups["number_of_labels"] == 1).sum()
)

print(
    "Same-feature / conflicting-label groups:",
    (exact_feature_groups["number_of_labels"] > 1).sum()
)

print("\nGroup sizes:")
print(exact_feature_groups["group_size"].value_counts())

In [25]:
# Verify repeated feature vectors using the actual 52 feature columns

repeated_rows = df.loc[
    repeated_feature_mask,
    feature_cols + [label_col]
].copy()

exact_feature_groups = (
    repeated_rows
    .groupby(feature_cols, dropna=False)[label_col]
    .agg(
        group_size="size",
        number_of_labels="nunique"
    )
    .reset_index()
)

print(
    "Repeated feature-vector groups:",
    len(exact_feature_groups)
)

print(
    "Same-feature / same-label groups:",
    (exact_feature_groups["number_of_labels"] == 1).sum()
)

print(
    "Same-feature / conflicting-label groups:",
    (exact_feature_groups["number_of_labels"] > 1).sum()
)

print("\nGroup sizes:")
print(exact_feature_groups["group_size"].value_counts())

Repeated feature-vector groups: 858
Same-feature / same-label groups: 161
Same-feature / conflicting-label groups: 697

Group sizes:
group_size
2    858
Name: count, dtype: int64


In [26]:
# Create a table containing only the conflicting records

conflicting_records = df.loc[conflicting_mask].copy()

# Preserve their original dataframe row numbers for auditing
conflicting_records.insert(
    0,
    "Original_Row_Index",
    conflicting_records.index
)

conflicting_records["Feature_Hash"] = (
    feature_hash[conflicting_mask].values
)

print("Conflicting records:", len(conflicting_records))

Conflicting records: 1394


In [27]:
# Show which class-label combinations conflict

conflict_label_pairs = (
    conflicting_records
    .groupby("Feature_Hash")[label_col]
    .agg(lambda x: " <-> ".join(sorted(set(x.astype(str)))))
    .value_counts()
)

conflict_label_pairs

Attack Type
Normal Traffic <-> Port Scanning    564
DoS <-> Normal Traffic              130
DDoS <-> Normal Traffic               3
Name: count, dtype: int64

In [28]:
conflicting_records[label_col].value_counts()

Attack Type
Normal Traffic    697
Port Scanning     564
DoS               130
DDoS                3
Name: count, dtype: int64

In [29]:
from pathlib import Path

audit_folder = data_file.parent / "V2_Data_Quality_Audit"
audit_folder.mkdir(exist_ok=True)

# Save conflicting records
conflicting_records.to_csv(
    audit_folder / "conflicting_label_records_V2.csv",
    index=False
)

# Save exact duplicate records
exact_duplicate_records = df.loc[
    exact_duplicate_mask
].copy()

exact_duplicate_records.insert(
    0,
    "Original_Row_Index",
    exact_duplicate_records.index
)

exact_duplicate_records.to_csv(
    audit_folder / "exact_duplicate_records_V2.csv",
    index=False
)

print("Audit files saved to:")
print(audit_folder)

Audit files saved to:
/Users/kevinvillon/Desktop/NOVA_IT691/Data_Version2/V2_Data_Quality_Audit


In [30]:
conflicting_records[label_col].value_counts()

Attack Type
Normal Traffic    697
Port Scanning     564
DoS               130
DDoS                3
Name: count, dtype: int64

In [31]:
# Create V3 by removing all conflicting-label records

df_v3 = df.loc[~conflicting_mask].copy()

print("Rows before removing conflicting records:", len(df))
print("Rows after removing conflicting records:", len(df_v3))
print("Conflicting rows removed:", len(df) - len(df_v3))

Rows before removing conflicting records: 2520751
Rows after removing conflicting records: 2519357
Conflicting rows removed: 1394


In [32]:
# Remove redundant exact duplicates while keeping one valid copy

before_duplicates = len(df_v3)

df_v3 = df_v3.drop_duplicates(
    subset=df.columns,
    keep="first"
).copy()

after_duplicates = len(df_v3)

print("Extra exact duplicates removed:",
      before_duplicates - after_duplicates)

print("Final V3 rows:", len(df_v3))

Extra exact duplicates removed: 161
Final V3 rows: 2519196


In [33]:
# Verify there are no exact duplicate rows

print(
    "Remaining exact duplicate rows:",
    df_v3.duplicated(keep=False).sum()
)

Remaining exact duplicate rows: 0


In [34]:
# Verify no identical feature vectors remain

remaining_feature_duplicates = df_v3.duplicated(
    subset=feature_cols,
    keep=False
).sum()

print(
    "Remaining repeated feature-vector rows:",
    remaining_feature_duplicates
)

Remaining repeated feature-vector rows: 0


In [35]:
v3_class_distribution = pd.DataFrame({
    "Count": df_v3[label_col].value_counts(),
    "Percentage":
        df_v3[label_col].value_counts(normalize=True) * 100
})

v3_class_distribution

,Count,Percentage
Attack Type,,
Normal Traffic,2094199,83.129657
DoS,193615,7.685587
DDoS,128011,5.081423
Port Scanning,90130,3.577729
Brute Force,9150,0.363211
Web Attacks,2143,0.085067
Bots,1948,0.077326


In [36]:
v3_file = (
    data_file.parent /
    "cicids2017_cleaned_WHOLE_SET_V3.csv"
)

df_v3.to_csv(
    v3_file,
    index=False
)

print("V3 dataset saved to:")
print(v3_file)

V3 dataset saved to:
/Users/kevinvillon/Desktop/NOVA_IT691/Data_Version2/cicids2017_cleaned_WHOLE_SET_V3.csv
